In [ ]:
!pip install cvxopt

### SVM and kernel definitions

In [ ]:
# Define ker functions for SVM
import cvxopt
from cvxopt import matrix, solvers
import numpy as np

def create_gaussian_ker(gamma=0.001):
    def ker(x, z):
        xx = (x**2).sum(axis=1).reshape(-1, 1) @ np.ones((1, z.shape[0]))
        zz = (z**2).sum(axis=1).reshape(-1, 1) @ np.ones((1, x.shape[0]))
        return np.exp(-gamma * (xx + zz.T - 2 * x @ z.T))
    return ker

def lin_ker(x, z):
    return np.dot(x, z.T)

In [ ]:
# SVM Classifier class definition
class SupportVectorMachine:
    '''
    Binary Classifier using Support Vector Machine
    '''
    def __init__(self):
        """
        Initializes the model's parameters to None. They will be learned in the `fit` method.
        """
        self.ker = None
        self.support_vectors = None
        self.sv_labels = None
        self.alphas_sv = None
        self.b = None
        self.w = None
        
    def fit(self, X, y, ker='lin', C=1.0, gamma=0.001):
        '''
        Learn the parameters from the given training data by solving the dual QP problem.
        
        Args:
            X: np.array of shape (N, D) 
                where N is the number of samples and D is the flattened dimension of each image
                
            y: np.array of shape (N,)
                where N is the number of samples and y[i] is the class of the ith sample (0 or 1)
                
            ker: str
                The ker to be used. Can be 'lin' or 'gaussian'
                
            C: float
                The regularization parameter for the soft margin
                
            gamma: float
                The gamma parameter for the gaussian ker, ignored for the lin ker
        '''
        # Map labels from {0, 1} to {-1, 1} for the SVM formulation
        y_mapped = y.copy()
        y_mapped[y_mapped == 0] = -1
        
        m, n = X.shape

        # Compute ker matrix
        if ker == 'lin':
            self.ker = lin_ker
        elif ker == 'gaussian':
            self.ker = create_gaussian_ker(gamma)
        else:
            raise ValueError("Unsupported ker type")
        
        ker_mat = self.ker(X, X)

        # Set the Quadratic Programming (QP) problem parameters
        # P_ij = y_i * y_j * K(x_i, x_j)
        P = matrix(np.outer(y_mapped, y_mapped) * ker_mat, tc='d')
        # q is a vector of -1s
        q = matrix(-np.ones(m), tc='d')
        # Inequality constraints for the box: 0 <= alpha_i <= C
        # -alpha_i <= 0  and  alpha_i <= C
        G = matrix(np.vstack([-np.eye(m), np.eye(m)]), tc='d')
        h = matrix(np.hstack([np.zeros(m), C * np.ones(m)]), tc='d')
        # Equality constraint: sum(alpha_i * y_i) = 0
        A = matrix(y_mapped, (1, m), 'd')
        b = matrix(0.0, tc='d')
        # Solve QP problem
        solvers.options['show_progress'] = True
        solution = solvers.qp(P, q, G, h, A, b)

        # Extract alphas
        alphas = np.array(solution['x']).reshape(-1)
        
        # Identify support vectors
        sv_indices = alphas > 1e-5
        
        # Learn model parameters
        self.alphas_sv = alphas[sv_indices]
        self.support_vectors = X[sv_indices]
        self.sv_labels = y_mapped[sv_indices]

        # Calculate the bias term using the support vectors
        ind = np.arange(len(alphas))[sv_indices]
        self.b = np.mean([
            self.sv_labels[i] - np.sum(
                self.alphas_sv * self.sv_labels * ker_mat[ind[i], sv_indices]
            ) for i in range(len(self.alphas_sv))
        ])

        # Calculate the weight vector w for lin ker
        if self.ker == lin_ker:
            self.w = np.sum(
                self.alphas_sv[:, None] * self.sv_labels[:, None] * self.support_vectors, 
                axis=0
            )


    def predict(self, X):
        '''
        Predict the class of the input data using the learned parameters.
        
        Args:
            X: np.array of shape (N, D) 
                where N is the number of samples and D is the flattened dimension of each image
                
        Returns:
            np.array of shape (N,)
                where N is the number of samples and y[i] is the predicted class
                for the ith sample (0 or 1)
        '''
        if self.ker is None:
            raise RuntimeError("The model has not been fitted yet")

        # Calculate the decision function score
        if self.ker == lin_ker:
            scores = np.dot(X, self.w) + self.b
        else:
            K = self.ker(X, self.support_vectors)
            scores = np.sum(K * self.alphas_sv * self.sv_labels, axis=1) + self.b
            
        # Convert scores to {-1, 1} and then to {0, 1} classes
        preds = np.sign(scores)
        preds[preds == -1] = 0 
        
        return preds.astype(int)

### Data preprocessing and setup

In [ ]:
!pip install seaborn
!pip install scikit-learn

In [ ]:
import os
import numpy as np
from PIL import Image

In [ ]:
def load_and_preprocess_data(base_dir, class_map, target_size=(32, 32)):
    imgs = []
    labels = []
    sorted_classes = sorted(class_map.keys())
    
    for class_name in sorted_classes:
        label = class_map[class_name]
        class_dir = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_dir):
            print(f"Directory not found at {class_dir}")
            continue
            
        for image_name in sorted(os.listdir(class_dir)): 
            img_path = os.path.join(class_dir, image_name)
            try:
                with Image.open(img_path) as img:
                    img_resized = img.resize(target_size).convert('RGB')
                    img_array = np.array(img_resized)
                    img_flat = img_array.flatten() / 255.0
                    imgs.append(img_flat)
                    labels.append(label)
            except Exception as e:
                print(f"Error processing {img_path}")
                
    return np.array(imgs), np.array(labels)

In [ ]:
# Use 0 and 1 classes based on my entry number (2022MT11920)
bin_class_map = {'airplane': 0, 'automobile': 1}
all_class_names = sorted(['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'])
train_dir = 'train'
test_dir = 'test'
# Load and preprocess the binary data
X_train_bin, y_train_bin = load_and_preprocess_data(train_dir, bin_class_map)
X_test_bin, y_test_bin = load_and_preprocess_data(test_dir, bin_class_map)
print("Binary Classification:")
print(f"Shape of training data: {X_train_bin.shape}")
print(f"Shape of training labels: {y_train_bin.shape}")
print(f"Shape of testing data: {X_test_bin.shape}")
print(f"Shape of testing labels: {y_test_bin.shape}")
print(f"Unique labels in training data: {np.unique(y_train_bin)}")


### CVXOPT with Linear Kernel

In [ ]:
# Question 1
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Train our SVM Model
print("\nTraining model...")
svm_lin = SupportVectorMachine()
start_time = time.time()
svm_lin.fit(X_train_bin, y_train_bin, ker='lin', C=1.0)
end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds.")

# Analyze Support Vectors
num_sv = len(svm_lin.support_vectors)
sv_percent = 100 * num_sv / len(X_train_bin)
print(f"Number of support vectors: {num_sv}")
print(f"Percentage of training samples as SVs: {sv_percent:.2f}%")

# Calculate w, b, and test accuracy
w = svm_lin.w
b = svm_lin.b
y_pred_lin = svm_lin.predict(X_test_bin)
accuracy_lin = accuracy_score(y_test_bin, y_pred_lin)
print(f"Test accuracy: {100 * accuracy_lin:.2f}%")
print(f"Calculated w shape: {w.shape}")
print(f"Calculated b: {b:.4f}")

# Plot the top 5 support vectors
print("Plotting top 5 support vectors and the weight vector...")

# Find the indices of the top 5 largest alpha values
top_5_indices = np.argsort(svm_lin.alphas_sv)[-5:]

# Denormalize the support vectors for visualization
sv_denormalized = svm_lin.support_vectors * 255.0

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
fig.suptitle("lin SVM: Top 5 SVs & Weight Vector")

# Plot the 5 support vectors
for i, idx in enumerate(top_5_indices):
    img = sv_denormalized[idx].reshape(32, 32, 3)
    img = np.clip(img, 0, 255).astype(np.uint8)
    alpha_val = svm_lin.alphas_sv[idx]
    axes[i].imshow(img)
    axes[i].set_title(f"SV (α≈{alpha_val:.2f})")
    axes[i].axis('off')
    
# Plot the weight vector as an image
w_img = w.reshape(32, 32, 3)
w_img_scaled = (w_img - w_img.min()) / (w_img.max() - w_img.min())
axes[5].imshow(w_img_scaled)
axes[5].set_title("Weight Vector w")
axes[5].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### CVXOPT with Gaussian Kernel

In [ ]:
# Question 2
# Train SVM Model
print("\nTraining the model with gaussian kernel...")
svm_gaussian = SupportVectorMachine()
start_time = time.time()
svm_gaussian.fit(X_train_bin, y_train_bin, ker='gaussian', C=1.0, gamma=0.001)
end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds.")

# Analyze Support Vectors
num_sv_gaussian = len(svm_gaussian.support_vectors)
sv_percent_gaussian = 100 * num_sv_gaussian / len(X_train_bin)
print(f"Number of support vectors: {num_sv_gaussian}")
print(f"Percent of training samples as SVs: {sv_percent_gaussian:.2f}%")

# Compare with lin ker support vectors
matching_sv = 0
for sv in svm_gaussian.support_vectors:
    if any(np.array_equal(sv, lsv) for lsv in svm_lin.support_vectors):
        matching_sv += 1
print(f"Number of matching support vectors with lin ker: {matching_sv}")

# Test Accuracy
y_pred_gaussian = svm_gaussian.predict(X_test_bin)
accuracy_gaussian = accuracy_score(y_test_bin, y_pred_gaussian)
print(f"Test accuracy: {100 * accuracy_gaussian:.2f}%")

# Plot Top-5 Support Vectors
print("Plotting top 5 support vectors...")
# Find the indices of the top 5 largest alpha values
top_5_indices_gaussian = np.argsort(svm_gaussian.alphas_sv)[-5:]
# Denormalize the support vectors for visualization
sv_denormalized_gaussian = svm_gaussian.support_vectors * 255.0
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle("Gaussian SVM: Top 5 SVs")

for i, idx in enumerate(top_5_indices_gaussian):
    img = sv_denormalized_gaussian[idx].reshape(32, 32, 3)
    img = np.clip(img, 0, 255).astype(np.uint8)
    alpha_val = svm_gaussian.alphas_sv[idx]
    axes[i].imshow(img)
    axes[i].set_title(f"SV (α≈{alpha_val:.2f})")
    axes[i].axis('off')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()
# Compare Test Accuracy
print(f"Test accuracy comparison:")
print(f"Lin ker Accuracy: {100 * accuracy_lin:.2f}%")
print(f"Gaussian ker Accuracy: {100 * accuracy_gaussian:.2f}%")


### Using inbuilt SVM from sklearn

In [ ]:
# Question 3
from sklearn.svm import SVC
import time
# Linear kernel
print("Linear Kernel with sklearn SVC")
svm_sklearn_lin = SVC(kernel='linear', C=1.0)
start_time = time.time()
svm_sklearn_lin.fit(X_train_bin, y_train_bin)
end_time = time.time()
training_time_sklearn_lin = end_time - start_time
print(f"Training completed in {training_time_sklearn_lin:.2f} seconds.")
n_sv_sklearn_lin = len(svm_sklearn_lin.support_)
print(f"Number of support vectors: {n_sv_sklearn_lin}")
# Compare support vectors
matching_sv_lin = 0
for sv in svm_sklearn_lin.support_vectors_:
    if any(np.array_equal(sv, lsv) for lsv in svm_lin.support_vectors):
        matching_sv_lin += 1
print(f"Number of matching support vectors with our SVM: {matching_sv_lin}")
# Compare weight and bias
w_sklearn_lin = svm_sklearn_lin.coef_.flatten()
b_sklearn_lin = svm_sklearn_lin.intercept_[0]
print(f"w diff norm: {np.linalg.norm(w - w_sklearn_lin):.4f}")
print(f"b diff: {abs(b - b_sklearn_lin):.4f}")
# Test accuracy
y_pred_sklearn_lin = svm_sklearn_lin.predict(X_test_bin)
accuracy_sklearn_lin = accuracy_score(y_test_bin, y_pred_sklearn_lin)
print(f"Test accuracy: {100 * accuracy_sklearn_lin:.2f}%")
# Gaussian kernel
print("\nGaussian Kernel with sklearn SVC")
svm_sklearn_gaussian = SVC(kernel='rbf', C=1.0, gamma=0.001)
start_time = time.time()
svm_sklearn_gaussian.fit(X_train_bin, y_train_bin)
end_time = time.time()
training_time_sklearn_gaussian = end_time - start_time
print(f"Training completed in {training_time_sklearn_gaussian:.2f} seconds.")
n_sv_sklearn_gaussian = len(svm_sklearn_gaussian.support_)
print(f"Number of support vectors: {n_sv_sklearn_gaussian}")
# Compare support vectors
matching_sv_gaussian = 0
for sv in svm_sklearn_gaussian.support_vectors_:
    if any(np.array_equal(sv, gsv) for gsv in svm_gaussian.support_vectors):
        matching_sv_gaussian += 1
print(f"Number of matching support vectors with our SVM: {matching_sv_gaussian}")
# Test accuracy
y_pred_sklearn_gaussian = svm_sklearn_gaussian.predict(X_test_bin)
accuracy_sklearn_gaussian = accuracy_score(y_test_bin, y_pred_sklearn_gaussian)
print(f"Test accuracy: {100 * accuracy_sklearn_gaussian:.2f}%")
# Compare computational cost
print(f"Training time comparison:")
print(f"Our lin ker: {end_time - start_time:.2f} seconds")
print(f"sklearn lin ker: {training_time_sklearn_lin:.2f} seconds")
print(f"Our Gaussian ker: {end_time - start_time:.2f} seconds")
print(f"sklearn Gaussian ker: {training_time_sklearn_gaussian:.2f} seconds")

### One vs One SVM classifier (Ours)

In [ ]:
# Question 5
from itertools import combinations
from collections import Counter
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Load and preprocess the full dataset with all classes
full_class_map = {name: idx for idx, name in enumerate(all_class_names)}
X_train_full, y_train_full = load_and_preprocess_data(train_dir, full_class_map)
X_test_full, y_test_full = load_and_preprocess_data(test_dir, full_class_map)
print(f"\nTraining data shape: {X_train_full.shape}")

# One-vs-One SVM Classifier
class OneVsOneSVM:
    def __init__(self, C=1.0, gamma=0.001):
        self.C = C
        self.gamma = gamma
        self.classif = {}
        self.classes = None
        
    def fit(self, X, y):
        self.classes = np.unique(y)
        for (class1, class2) in combinations(self.classes, 2):
            print(f"Training classifier for classes {class1} vs {class2}")
            idx = np.where((y == class1) | (y == class2))[0]
            X_pair = X[idx]
            y_pair = y[idx]
            y_pair[y_pair == class1] = 0
            y_pair[y_pair == class2] = 1
            svm = SupportVectorMachine()
            svm.fit(X_pair, y_pair, ker='gaussian', C=self.C, gamma=self.gamma)
            self.classif[(class1, class2)] = svm
            
    def predict(self, X):
        votes = np.zeros((X.shape[0], len(self.classes)))
        for (class1, class2), svm in self.classif.items():
            preds = svm.predict(X)
            for i, pred in enumerate(preds):
                if pred == 0:
                    votes[i, class1] += 1
                else:
                    votes[i, class2] += 1
        final_preds = np.argmax(votes, axis=1)
        return final_preds
    
# Train our One-vs-One SVM
print("\nTraining One-vs-One SVM...")
ovo_svm = OneVsOneSVM(C=1.0, gamma=0.001)
start_time = time.time()
ovo_svm.fit(X_train_full, y_train_full)
end_time = time.time()
print(f"One-vs-One training completed in {end_time - start_time:.2f} seconds.")
# Predict on the test data
y_pred_ovo = ovo_svm.predict(X_test_full)
acc_ovo = accuracy_score(y_test_full, y_pred_ovo)
print(f"Test accuracy: {100 * acc_ovo:.2f}%")

# Confusion Matrix
conf_matrix = confusion_matrix(y_test_full, y_pred_ovo)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=all_class_names, yticklabels=all_class_names)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix for One-vs-One Gaussian SVM')
plt.show()


### One-vs-One SVM classifier (inbuilt)

In [ ]:
# sklearn One-vs-One SVM
print("sklearn One-vs-One SVM:")
svm_sklearn_ovo = SVC(kernel='rbf', C=1.0, gamma=0.001, decision_function_shape='ovo')
start_time = time.time()
svm_sklearn_ovo.fit(X_train_full, y_train_full)
end_time = time.time()
training_time_sklearn_ovo = end_time - start_time
print(f"Training completed in {training_time_sklearn_ovo:.2f} seconds.")
y_pred_sklearn_ovo = svm_sklearn_ovo.predict(X_test_full)
accuracy_sklearn_ovo = accuracy_score(y_test_full, y_pred_sklearn_ovo)
print(f"Test accuracy: {100 * accuracy_sklearn_ovo:.2f}%")
print(f"Training time comparison:")
print(f"Our One-vs-One Gaussian ker: {end_time - start_time:.2f} seconds")
print(f"sklearn One-vs-One Gaussian ker: {training_time_sklearn_ovo:.2f} seconds")
# Confusion Matrix for sklearn One-vs-One
conf_matrix_sklearn = confusion_matrix(y_test_full, y_pred_sklearn_ovo)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix_sklearn, annot=True, fmt='d', cmap='Greens',
            xticklabels=all_class_names, yticklabels=all_class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for sklearn One-vs-One Gaussian SVM')
plt.show()

### Misclassification analysis

In [ ]:
# Analyze misclassifications
missed_ind = np.where(y_test_full != y_pred_ovo)[0]
print(f"Number of misclassified samples: {len(missed_ind)}")
# Visualize 10 misclassified examples
num_to_display = min(10, len(missed_ind))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Misclassified Examples by Our One-vs-One SVM")
for i in range(num_to_display):
    idx = missed_ind[i]
    img = (X_test_full[idx] * 255).reshape(32, 32, 3).astype(np.uint8)
    true_label = all_class_names[y_test_full[idx]]
    pred_label = all_class_names[y_pred_ovo[idx]]
    axes[i // 5, i % 5].imshow(img)
    axes[i // 5, i % 5].set_title(f"True: {true_label}\nPred: {pred_label}")
    axes[i // 5, i % 5].axis('off')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Analyze misclassifications for sklearn One-vs-One
missed_ind_sklearn = np.where(y_test_full != y_pred_sklearn_ovo)[0]
print(f"Number of misclassified samples by sklearn: {len(missed_ind_sklearn)}")
# Visualize 10 misclassified examples by sklearn
num_to_display_sklearn = min(10, len(missed_ind_sklearn))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Misclassified Examples by sklearn One-vs-One SVM")
for i in range(num_to_display_sklearn):
    idx = missed_ind_sklearn[i]
    img = (X_test_full[idx] * 255).reshape(32, 32, 3).astype(np.uint8)
    true_label = all_class_names[y_test_full[idx]]
    pred_label = all_class_names[y_pred_sklearn_ovo[idx]]
    axes[i // 5, i % 5].imshow(img)
    axes[i // 5, i % 5].set_title(f"True: {true_label}\nPred: {pred_label}")
    axes[i // 5, i % 5].axis('off')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


### Cross-Validation analysis and final SVM model

In [ ]:
from sklearn.model_selection import KFold
C_vals = [1e-5, 1e-3, 1, 5, 10]
gamma = 0.001
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_accs = []
test_accs = []
for C in C_vals:
    fold_accs = []
    for train_index, val_index in kf.split(X_train_full):
        X_train_fold, X_val_fold = X_train_full[train_index], X_train_full[val_index]
        y_train_fold, y_val_fold = y_train_full[train_index], y_train_full[val_index]
        svm = SVC(kernel='rbf', C=C, gamma=gamma)
        svm.fit(X_train_fold, y_train_fold)
        y_val_pred = svm.predict(X_val_fold)
        fold_acc = accuracy_score(y_val_fold, y_val_pred)
        fold_accs.append(fold_acc)
    mean_cv_acc = np.mean(fold_accs)
    cv_accs.append(mean_cv_acc)
    # Evaluate on the test data
    y_test_pred = svm.predict(X_test_full)
    test_acc = accuracy_score(y_test_full, y_test_pred)
    test_accs.append(test_acc)
    print(f"C={C}: 5-fold CV Accuracy={mean_cv_acc:.4f}, Test Accuracy={test_acc:.4f}")
# Plot the accuracies
plt.figure(figsize=(10, 6))
plt.plot(C_vals, cv_accs, marker='o', label='5-fold CV Accuracy')
plt.plot(C_vals, test_accs, marker='s', label='Test Accuracy')
plt.xscale('log')
plt.xlabel('C value (log scale)')
plt.ylabel('Accuracy')
plt.title('5-fold Cross-Validation and Test Accuracy vs C value')
plt.legend()
plt.grid(True)
plt.show()
# Find the best C value
best_C_index = np.argmax(cv_accs)
best_C = C_vals[best_C_index]
print(f"Best C value based on 5-fold CV accuracy: {best_C}")
# Train final model with the best C value on the entire training data
final_svm = SVC(kernel='rbf', C=best_C, gamma=gamma)
final_svm.fit(X_train_full, y_train_full)
y_final_test_pred = final_svm.predict(X_test_full)
final_test_acc = accuracy_score(y_test_full, y_final_test_pred)
print(f"Final Test Accuracy with C={best_C}: {final_test_acc:.4f}")
